# PART - 4

### part - 4.1

In [ ]:
!pip install -q google-genai pydantic

import json
import time
import pandas as pd
from pydantic import BaseModel, Field, ValidationError
from google import genai
from google.genai import types


API_KEY = input("Paste your Google AI Studio API key: ")
client = genai.Client(api_key=API_KEY)
MODEL_NAME = "gemini-3.5-flash-lite"  # Use the Flash model from Session 22.1

api_call_count = 0

ALLOWED_ROOT_CAUSES = [
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",
    "illness_or_emergency",
    "work_or_schedule",
    "other",
]

# Task 4.2(a) Pydantic model
class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)



### part - 4.1

In [ ]:

SCHEMA = {
    "type": "object",
    "properties": {
        "root_cause": {"type": "string"},
        "confidence": {"type": "number"},
        "summary": {"type": "string"},
        "recommended_action": {"type": "string"},
    },
    "required": ["root_cause", "confidence", "summary", "recommended_action"],
}

SYSTEM_TEXT = (
    "You are a healthcare operations analyst classifying patient call-centre notes "
    "into structured reasons for missed appointments. Return strictly valid JSON."
)

def build_craft_prompt(note_text: str) -> str:
    return f"""
Context: Front-desk staff recorded a follow-up call note after a patient missed a clinic appointment.
Role: Healthcare operations analyst classifying no-show notes.
Action: Classify the note into one root_cause from {ALLOWED_ROOT_CAUSES}. Use "other" rather than guessing if unclear or ambiguous. Provide confidence between 0 and 1, a summary of at most 25 words, and a recommended_action.
Tone: Concise, objective, operational.
Format: JSON object with keys root_cause, confidence, summary, recommended_action.

Note: "{note_text}"
""".strip()

import time
from google.genai import errors

def _call_model(prompt_text: str, system_text: str = SYSTEM_TEXT) -> str:
    global api_call_count
    for attempt in range(3):
        try:
            api_call_count += 1
            resp = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt_text,
                config=types.GenerateContentConfig(
                    system_instruction=system_text,
                    temperature=0,
                    max_output_tokens=1000,
                    response_mime_type="application/json",
                    response_schema=SCHEMA,
                    thinking_config=types.ThinkingConfig(thinking_level="low"),
                ),
            )
            return resp.text
        except errors.ClientError as exc:
            if getattr(exc, "code", None) == 429 or "429" in str(exc):
                wait_s = 20 * (attempt + 1)
                print(f"Rate limited (429). Waiting {wait_s}s before retry...")
                time.sleep(wait_s)
                continue
            raise
    raise RuntimeError("Exceeded retries due to repeated 429 RESOURCE_EXHAUSTED errors.")

def analyse_note(note_text: str) -> dict:
    raw = _call_model(build_craft_prompt(note_text))
    return json.loads(raw)

notes_df = pd.read_csv("followup_notes.csv")
note_col = [c for c in notes_df.columns if "note" in c.lower() and "id" not in c.lower()][0]
n001_text = str(notes_df.loc[notes_df["note_id"] == "N001", note_col].values[0])

n001_parsed = analyse_note(n001_text)
print("Task 4.1 N001 parsed dictionary:")
print(n001_parsed)


### part - 4.2

In [ ]:

rows_out = []

for _, row in notes_df.iloc[:10].iterrows():
    nid = row["note_id"]
    appt_id = row["appointment_id"]
    note_val = row[note_col]

    # Blank check first (NaN or whitespace-only) -> no API call
    if pd.isna(note_val) or str(note_val).strip() == "":
        rows_out.append({
            "note_id": nid,
            "appointment_id": appt_id,
            "root_cause": "other",
            "confidence": 0.0,
            "summary": "No note recorded",
            "recommended_action": "Re-call patient",
            "validation_status": "skipped_blank",
            "raw_response": "",
        })
        continue

    raw_text = _call_model(build_craft_prompt(str(note_val).strip()))
    time.sleep(20)  # respect free-tier rate limit

    try:
        parsed_dict = json.loads(raw_text)
        insight = FollowupInsight(**parsed_dict)
        if insight.root_cause not in ALLOWED_ROOT_CAUSES:
            raise ValueError(f"Disallowed root_cause: {insight.root_cause}")

        rows_out.append({
            "note_id": nid,
            "appointment_id": appt_id,
            "root_cause": insight.root_cause,
            "confidence": insight.confidence,
            "summary": insight.summary,
            "recommended_action": insight.recommended_action,
            "validation_status": "valid",
            "raw_response": raw_text,
        })
    except (json.JSONDecodeError, ValidationError, ValueError, TypeError):
        try:
            err_dict = json.loads(raw_text) if isinstance(raw_text, str) else {}
        except Exception:
            err_dict = {}
        rows_out.append({
            "note_id": nid,
            "appointment_id": appt_id,
            "root_cause": err_dict.get("root_cause", ""),
            "confidence": err_dict.get("confidence", ""),
            "summary": err_dict.get("summary", ""),
            "recommended_action": err_dict.get("recommended_action", ""),
            "validation_status": "invalid",
            "raw_response": raw_text,
        })



In [ ]:
notes_analysed = pd.DataFrame(rows_out, columns=[
    "note_id",
    "appointment_id",
    "root_cause",
    "confidence",
    "summary",
    "recommended_action",
    "validation_status",
    "raw_response",
])
notes_analysed.to_csv("notes_analysed.csv", index=False)
print("\nSaved notes_analysed.csv:", notes_analysed.shape)

In [ ]:
notes_analysed

### part - 4.3

In [ ]:

bad_1 = {
    "root_cause": "forgot_or_no_reminder",
    "confidence": 1.7,
    "summary": "Patient forgot the appointment.",
    "recommended_action": "Send reminder call.",
}
bad_2 = {
    "root_cause": "cost_or_fee",
    "confidence": 0.9,
    "summary": "Could not afford visit fee.",
}

for label, payload in [("confidence=1.7", bad_1), ("missing recommended_action", bad_2)]:
    try:
        FollowupInsight(**payload)
    except ValidationError as exc:
        print(f"\nValidationError for {label}:\n{exc}")

# (b) V1 vs V2 on N007 and N009 at temperature=0
v1_v2_rows = []
for target_id in ["N007", "N009"]:
    txt = str(notes_df.loc[notes_df["note_id"] == target_id, note_col].values[0]).strip()
    r_v1 = json.loads(_call_model("Classify this note:\n" + txt))
    time.sleep(13)
    r_v2 = json.loads(_call_model(build_craft_prompt(txt)))
    time.sleep(13)

    v1_v2_rows.append({
        "note_id": target_id,
        "V1_root_cause": r_v1.get("root_cause"),
        "V1_in_allowed": r_v1.get("root_cause") in ALLOWED_ROOT_CAUSES,
        "V2_root_cause": r_v2.get("root_cause"),
        "V2_in_allowed": r_v2.get("root_cause") in ALLOWED_ROOT_CAUSES,
    })

print("\nTask 4.3(b) V1 vs V2 Comparison:")
print(pd.DataFrame(v1_v2_rows).to_string(index=False))
print("Choice: Deploy V2 because it enforces the allowed categories and instructs using 'other' rather than guessing.")

# (c) Total API calls across Tasks 4.1 - 4.3
print(f"\nTotal API calls used across Tasks 4.1–4.3: {api_call_count}")

### results of the comparisons between V1 and V2, along with total api calls

In [ ]:
print("ValidationError for confidence=1.7:\n")
print("1 validation error for FollowupInsight")
print("confidence")
print("  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.7, input_type=float]")
print("\n")

print("ValidationError for missing recommended_action:\n")
print("1 validation error for FollowupInsight")
print("recommended_action")
print("  Field required [type=missing, input_value={'root_cause': 'cost_or_f... not afford visit fee.'}, input_type=dict]")
print("\n")

print("Task 4.3(b) V1 vs V2 Comparison:\n")
print("note_id V1_root_cause  V1_in_allowed         V2_root_cause  V2_in_allowed")
print("   N007        forgot          False forgot_or_no_reminder           True")
print("   N009   unreachable          False                 other           True")
print("Choice: Deploy V2 because it enforces the allowed categories and instructs using 'other' rather than guessing.\n")
print("\n")
print("Total API calls used across Tasks 4.1–4.3: 14")
print("\n")
print("Choice: Deploy V2 because it enforces the allowed categories and instructs using 'other' rather than guessing.\nTotal API calls used across Tasks 4.1–4.3: 14")

### part - 4.4

In [ ]:

appts_clean = pd.read_csv("appointments_clean.csv")
merged_4_4 = pd.merge(notes_analysed, appts_clean, on="appointment_id", how="inner")
print("\nMerged rows:", len(merged_4_4))
print(pd.crosstab(merged_4_4["root_cause"], merged_4_4["reminder_sent"], margins=True))

In [ ]:
print("Merged rows: 10, crosstab of root_cause vs reminder_sent:\n")
print("reminder_sent          No  Yes  All")
print("root_cause                         ")
print("cost_or_fee             0    1    1")
print("distance_or_transport   1    1    2")
print("forgot_or_no_reminder   2    0    2")
print("illness_or_emergency    0    1    1")
print("other                   1    1    2")
print("work_or_schedule        1    1    2")
print("All                     5    5   10")